<a href="https://colab.research.google.com/github/usman-nadeembajwa/Doordash/blob/main/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# W07 — Content Action Playbook
### Turning my validated model output into a human-reviewed set of recommendations

**What only I can fill in, marked below:**
- `SCORED_DATA_PATH` and column names in the CONFIG block — my actual validated output
  from W05/W06 (the honest-split model's scores per page)
- The archetype definitions and thresholds in Section 2 — these should reflect what I
  actually saw in my data, not generic guesses
- The no-go list in Section 3 — specific to my dataset/site categories

**Order (matches the assignment):**
1. Ranked actions + reason codes
2. Intended use and limits
3. Human review + the no-go list
4. Monitoring / retrain triggers
5. Exports for the paper
6. Self-check


## Setup

In [1]:
import pandas as pd
import numpy as np
import json as _json
import matplotlib.pyplot as plt

# ====== CONFIG — update to match my real W06 output ======
SCORED_DATA_PATH = "work/outputs/w06_scored_pages.csv"   # my honest-split model's scored output
SCORE_COL = "score"            # model's predicted probability / opportunity score
LABEL_COL = "label"            # ground-truth label, if available for the historical set
TRAFFIC_COL = "clicks"         # a size/value proxy — traffic, clicks, or impressions
POSITION_COL = "position"      # average search position, if available
DATE_COL = "date"              # last-updated / last-crawled date, if available
PAGE_ID_COL = "page_id"        # unique page/url identifier
FIGURES_DIR = "work/figures/"
OUTPUTS_DIR = "work/outputs/"
# ===========================================================

df = pd.read_csv(SCORED_DATA_PATH)
print(df.shape)
df.head()


FileNotFoundError: [Errno 2] No such file or directory: 'work/outputs/w06_scored_pages.csv'

## 1. Ranked Actions + Reason Codes

Rank pages by score, then attach a **reason code** to each — a short, human-readable
statement of *why* this page scored the way it did, built from the features that moved the
score most. This is what turns "the model said 0.83" into something a content editor can
actually act on.

In [ ]:
# Identify candidate feature columns (numeric, excluding score/label/id columns)
exclude = {SCORE_COL, LABEL_COL, PAGE_ID_COL, DATE_COL}
feature_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in exclude]
print(f"Using {len(feature_cols)} features for reason codes: {feature_cols}")

# Z-score each feature so "unusually high/low" is comparable across features of different scales
z = (df[feature_cols] - df[feature_cols].mean()) / df[feature_cols].std(ddof=0)

def top_reason_codes(row_z, n=2):
    """Return the n features furthest from the population average for this row,
    as plain-language reason strings."""
    ranked = row_z.abs().sort_values(ascending=False).index[:n]
    reasons = []
    for feat in ranked:
        direction = "unusually high" if row_z[feat] > 0 else "unusually low"
        reasons.append(f"{feat} is {direction} vs. the typical page")
    return "; ".join(reasons)

df["reason_code"] = z.apply(top_reason_codes, axis=1)
ranked = df.sort_values(SCORE_COL, ascending=False).reset_index(drop=True)
ranked[[PAGE_ID_COL, SCORE_COL, "reason_code"]].head(15)


### Archetype → Action mapping

Define page archetypes from the data itself, then map each to one recommended action.
**Update the thresholds below after actually looking at the distribution of `score`,
`TRAFFIC_COL`, and `POSITION_COL` in my data** — these starting cut points are placeholders,
not tuned values.

In [ ]:
# TODO: replace these placeholder thresholds after inspecting my actual distributions
HIGH_SCORE = df[SCORE_COL].quantile(0.75)
LOW_SCORE = df[SCORE_COL].quantile(0.25)
HIGH_TRAFFIC = df[TRAFFIC_COL].quantile(0.75) if TRAFFIC_COL in df.columns else None

def archetype(row):
    high_value = HIGH_TRAFFIC is not None and row[TRAFFIC_COL] >= HIGH_TRAFFIC
    if row[SCORE_COL] >= HIGH_SCORE and high_value:
        return "Declining — high traffic value"
    elif row[SCORE_COL] >= HIGH_SCORE:
        return "Declining — low traffic value"
    elif row[SCORE_COL] <= LOW_SCORE:
        return "Stable / low opportunity"
    else:
        return "Watch — mid-range signal"

ARCHETYPE_ACTION_MAP = {
    "Declining — high traffic value": "Priority refresh — rewrite/update content this sprint",
    "Declining — low traffic value": "Batch refresh or consolidate with a related page",
    "Watch — mid-range signal": "No action yet — re-score next cycle before deciding",
    "Stable / low opportunity": "No action",
}

df["archetype"] = df.apply(archetype, axis=1)
df["recommended_action"] = df["archetype"].map(ARCHETYPE_ACTION_MAP)

df["archetype"].value_counts()


### Decay / refresh insight

A short, honest read of what actually separates declining pages from stable ones in this
data — write 2–3 sentences here once the numbers below are in, using observed/measured
language, not causal claims.

In [ ]:
decay_summary = df.groupby("archetype")[feature_cols].mean()
decay_summary


**TODO — write my actual read of the table above.** Example structure: "Pages in the
declining/high-value archetype were observed to have [X] notably higher/lower than stable
pages, most consistently on [feature]. This is directional, not causal — it describes what
this scored dataset looked like, not a guarantee about any individual page."

## 2. Intended Use and Limits

**Intended use:** this playbook is decision-support for a content team — a ranked starting
point for prioritizing manual review, not an instruction to auto-publish changes.

**Limits (fill in from my actual W06 validation audit):**
- Precision@K under an honest split was measured at [X] — meaning roughly [X]% of the
  top-ranked pages, in my test set, were true positives. That is not a promise for every
  future run.
- The model has not been evaluated on [categories not represented in training data —
  e.g. brand-new page types, non-English content, etc. — state my actual gaps].
- Reason codes describe correlation with the score, not proof that a feature *causes* decline.


## 3. Human Review + No-Go List

**Human review is required before acting on any recommendation.** This model produces a
prioritized queue, not autonomous publishing decisions.

### No-go list — pages that should NEVER be auto-actioned regardless of score
Fill in categories specific to my dataset — starting examples below.

In [ ]:
NO_GO_CATEGORIES = [
    "Legal, compliance, or regulatory pages",
    "Pages under active legal/PR review",
    "Recently published pages (< 30 days old) — too little signal yet",
    "Pages tagged as evergreen/reference material regardless of score",
    # TODO: add categories specific to my actual dataset
]
for c in NO_GO_CATEGORIES:
    print("-", c)


### Human-review checklist (before any recommended action is executed)

- [ ] Confirm the page isn't on the no-go list above
- [ ] Confirm the reason code makes sense on manual read of the actual page
- [ ] Confirm traffic/value data is current, not stale
- [ ] For "priority refresh" actions specifically: assign to a writer, don't auto-publish


## 4. Cost / Value and Monitoring / Retrain Triggers

**Cost/value framing** — a lightweight way to prioritize within the ranked queue, not just
by score:

In [ ]:
def cost_tier(row):
    # TODO: replace with real cost signals if I have them (word count, page complexity, etc.)
    if row["archetype"] == "Declining — high traffic value":
        return "High value / worth the effort"
    elif row["archetype"] == "Declining — low traffic value":
        return "Lower value — batch or deprioritize if resourcing is tight"
    else:
        return "N/A"

df["cost_value_note"] = df.apply(cost_tier, axis=1)
df[["archetype", "cost_value_note"]].drop_duplicates()


**Monitoring / retrain triggers** — when this model's output should be treated with more
suspicion, or retrained:

- Re-score on a fixed cadence (e.g. monthly) — don't treat one score as permanent
- If the honest-split precision on a fresh evaluation batch drops meaningfully below the
  measured baseline from my W06 audit, flag for review before trusting new recommendations
- If the input data's feature distributions shift notably from what the model was trained
  on (e.g. a new content type is introduced), retrain rather than extrapolate
- If reason codes start clustering on a feature that later turns out to be unreliable/
  leaky, pull that feature and re-validate


## 5. Exports for the Paper

Per the assignment: the ranked queue CSV stays **out of git** (data files are blocked by the
CI leak-guard) — the notebook regenerates it on demand. Figures and metrics JSON get
committed since they're the receipts the paper's numbers trace back to.

In [ ]:
import os
os.makedirs(OUTPUTS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

export_cols = [PAGE_ID_COL, SCORE_COL, "archetype", "recommended_action", "reason_code", "cost_value_note"]
export_cols = [c for c in export_cols if c in df.columns or c in ["archetype", "recommended_action", "reason_code", "cost_value_note"]]

ranked_queue = df.sort_values(SCORE_COL, ascending=False)[export_cols]
ranked_queue.to_csv(f"{OUTPUTS_DIR}w07_ranked_action_queue.csv", index=False)
print(f"Saved: {OUTPUTS_DIR}w07_ranked_action_queue.csv (not committed — regenerated by this notebook)")

# Figure: archetype distribution — this one DOES get committed
fig, ax = plt.subplots(figsize=(7, 4))
df["archetype"].value_counts().plot(kind="barh", ax=ax, color="#2f5233")
ax.set_title("Pages by archetype")
ax.set_xlabel("Count")
plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}w07_archetype_distribution.png", dpi=150)
plt.show()

# Metrics JSON — committed, this is what the paper's numbers trace back to
metrics = {
    "archetype_counts": df["archetype"].value_counts().to_dict(),
    "high_score_threshold": float(HIGH_SCORE),
    "low_score_threshold": float(LOW_SCORE),
}
with open(f"{OUTPUTS_DIR}w07_metrics.json", "w") as f:
    _json.dump(metrics, f, indent=2)
print("Saved metrics JSON.")


## 6. Self-Check

- [ ] Ranked actions with reason codes are in place, built from actual feature contributions
- [ ] Archetype → action mapping is grounded in my real score/traffic distributions, not placeholder quantiles
- [ ] Decay/refresh insight is written in observed/measured language
- [ ] Intended use and limits reflect my actual W06 validation numbers
- [ ] No-go list and human-review checklist are specific to my dataset
- [ ] Cost/value and monitoring/retrain triggers are filled in, not left as TODOs
- [ ] Ranked queue CSV exports to `work/outputs/` (untracked); figures committed to `work/figures/`; metrics JSON committed

**Once all boxes are true:** commit to `work/notebooks/w07_action_playbook.ipynb`, commit the
figures/metrics, then submit the repo URL on the assignment card.
